# Lesson 01 - AIエージェント入門

**初心者のためのAIエージェント**コースの最初のレッスンへようこそ！

**AIエージェント**とは、大規模言語モデル(LLM)を推論エンジンとして使用し、APIの呼び出し、データベースのクエリ、コードの実行など、現実世界で*アクション*を起こしてユーザーの代わりに目標を達成できるプログラムのことです。

このノートブックでは、最初のエージェント、休暇先をおすすめする**トラベルエージェント**を作成します。その過程で以下を学びます：

1. **Microsoft Agent Framework**を使ってAzure AI Foundry Agent Serviceに接続する方法。
2. エージェントに**ツール**を与える—エージェントが呼び出せる普通のPython関数を用意する。
3. エージェントを実行し、その応答を検査する方法。
4. エージェントの応答をトークンごとにストリーム表示する方法。


## セットアップ

このノートブックを実行する前に、以下を確認してください：

1. **Azure AI Foundry プロジェクト** とチャットモデルがデプロイされていること（例：`gpt-4o-mini`）。
2. **Azure CLI でログイン済みであること** — ターミナルで `az login` を実行してください。
3. **必要な環境変数を設定していること：**
   - `AZURE_AI_PROJECT_ENDPOINT` — ご自身の Azure AI Foundry プロジェクトのエンドポイント。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — デプロイされたモデルの名前。

以下のセルは必要なPythonパッケージをインストールします。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 最初のエージェントを作成する

エージェントには2つのものが必要です。

- **エージェントが*誰であるか*と*どのように振る舞うか*を指示する**（システムプロンプト）。
- **ツール** — エージェントが情報を取得したり操作を行ったりするために呼び出せる、`@tool`デコレーターが付けられたPython関数。

以下では、人気のある旅行先のリストを返す簡単なツールを定義します。エージェントは、ユーザーが旅行のおすすめを求めたときにこのツールを使用します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ストリーミング応答

よりインタラクティブな体験のために、エージェントの応答を**ストリーミング**することができます。完全な返信を待つのではなく、エージェントは生成されるテキストのチャンクを逐次返します。これは、リアルタイムで出力を表示したいチャットインターフェースで特に有用です。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

このレッスンでは以下のことを学びました：

- `AzureAIProjectAgentProvider` を使って Azure AI Foundry Agent Service に接続する **プロバイダーを作成する** 方法。
- エージェントが Python 関数を呼び出せるように、`@tool` デコレーターを使って **ツールを定義する** 方法。
- ユーザーメッセージでエージェントを **起動し、その応答を出力する** 方法。
- リアルタイム出力のために、**レスポンスをストリーミングする** 方法。

次のレッスンでは、エージェントフレームワークをより深く探求し、エージェントにより強力なツールや多段階推論機能を持たせる方法を学びます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されました。正確性を期しておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があります。正式な情報源としては、原文（原言語で記載された文書）を参照してください。重要な情報については、専門の人間による翻訳を推奨いたします。本翻訳の利用により生じた誤解や誤訳に関して、当方は一切責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
